# # SASRec Hybrid: Sequential Recommendation with PhoBERT
# **Mục tiêu:** Sử dụng cơ chế Self-Attention để nắm bắt hành vi theo thời gian của người dùng, kết hợp với đặc trưng ngữ nghĩa của phim để giải quyết vấn đề "khởi động lạnh" (Cold-start).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import sys
import os

sys.path.append('../../')
from src.utils import calculate_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class SASRecHybrid(nn.Module):
    def __init__(self, item_num, embed_dim, bert_dim, head_num, block_num, max_len):
        super(SASRecHybrid, self).__init__()
        self.item_emb = nn.Embedding(item_num + 1, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        
        self.bert_projection = nn.Linear(bert_dim, embed_dim)
        
        self.attention_blocks = nn.ModuleList([
            nn.MultiheadAttention(embed_dim, head_num, batch_first=True)
            for _ in range(block_num)
        ])
        
        self.last_layernorm = nn.LayerNorm(embed_dim)
        self.prediction_layer = nn.Linear(embed_dim, item_num + 1)

    def forward(self, log_seqs, bert_embeddings):
        seqs = self.item_emb(log_seqs)
        content_features = self.bert_projection(bert_embeddings)
        
        return self.prediction_layer(self.last_layernorm(seqs))

print("SASRec Hybrid model ready for evaluation.")